In [18]:
# import kagglehub
# kagglehub.login()

# # export KAGGLE_API_TOKEN=KGAT_be96b99b82783874376c258e41c96de6

# # Download latest version
# path = kagglehub.competition_download('bt-4012-competition-2026')

# print("Path to competition files:", path)

In [19]:
# %pip install xgboost scikit-learn

In [20]:
import os
import pandas as pd
import numpy as np

data_dir = r"C:\Users\regan\.cache\kagglehub\competitions\bt-4012-competition-2026"
print("Files:", os.listdir(data_dir))

# Read the core files (adjust filenames if slightly different in the directory listing)
train_df = pd.read_csv(os.path.join(data_dir, "train.csv"))
test_df = pd.read_csv(os.path.join(data_dir, "test.csv"))
edges_df = pd.read_csv(os.path.join(data_dir, "txs_edgelist.csv")) # Contains txId1 -> txId2

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"Edges shape: {edges_df.shape}")

Files: ['sample_submission.csv', 'test.csv', 'train.csv', 'txs_edgelist.csv']
Train shape: (141772, 168)
Test shape:  (15329, 168)
Edges shape: (234355, 2)


In [21]:
features = [col for col in train_df.columns if col not in ['txId', 'label']]
X = train_df[features].copy()
y = train_df['label']
X_test = test_df[features].copy()

print(f'Features used: {X.shape[1]}')
print(f'Labeled rows: {y.notna().sum()}')
print(f'Unlabeled rows available for pseudo-labeling: {y.isna().sum()}')

Features used: 166
Labeled rows: 31235
Unlabeled rows available for pseudo-labeling: 110537


In [22]:
cutoff_step = train_df['time_step'].quantile(0.75)

labeled_mask = y.notna()
train_mask = (train_df['time_step'] <= cutoff_step) & labeled_mask
val_mask = (train_df['time_step'] > cutoff_step) & labeled_mask
unlabeled_train_mask = (train_df['time_step'] <= cutoff_step) & (~labeled_mask)

X_tr, y_tr = X.loc[train_mask], y.loc[train_mask].astype(int)
X_val, y_val = X.loc[val_mask], y.loc[val_mask].astype(int)
X_unlabeled_train = X.loc[unlabeled_train_mask]

print(f'Labeled train rows: {len(X_tr)}')
print(f'Labeled validation rows: {len(X_val)}')
print(f'Unlabeled rows eligible for pseudo-labeling: {len(X_unlabeled_train)}')

Labeled train rows: 23606
Labeled validation rows: 7629
Unlabeled rows eligible for pseudo-labeling: 85344


In [23]:
scale_pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()

In [24]:
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

pos_ratio = (y_tr == 0).sum() / (y_tr == 1).sum()

model = LGBMClassifier(
    objective='binary',
    metric='auc',
    boosting_type='gbdt',
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    scale_pos_weight=pos_ratio,
    random_state=42,
    n_jobs=-1
)

# Stage 1: train only on genuinely labeled rows.
model.fit(
    X_tr, y_tr,
    eval_set=[(X_tr, y_tr), (X_val, y_val)],
    eval_names=['train', 'val'],
    eval_metric='auc',
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=50)
    ]
)

# Stage 2: assign only high-confidence labels to blank rows.
unlabeled_probs = model.predict_proba(X_unlabeled_train)[:, 1]
confidence = np.maximum(unlabeled_probs, 1 - unlabeled_probs)
pseudo_mask = confidence >= 0.95

X_pseudo = X_unlabeled_train.loc[pseudo_mask]
y_pseudo = pd.Series((unlabeled_probs[pseudo_mask] >= 0.5).astype(int), index=X_pseudo.index)

X_aug = pd.concat([X_tr, X_pseudo])
y_aug = pd.concat([y_tr, y_pseudo])

print(f'High-confidence pseudo-labels added: {len(X_pseudo)}')
print(f'Pseudo-label positives: {int(y_pseudo.sum())}')

# Refit using real labels plus the selected pseudo-labels.
model.fit(
    X_aug, y_aug,
    eval_set=[(X_aug, y_aug), (X_val, y_val)],
    eval_names=['train_augmented', 'val'],
    eval_metric='auc',
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=50)
    ]
)

val_preds = model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, val_preds)
print(f'\n---> Temporal Validation ROC-AUC: {val_auc:.5f}')

# Keep every genuinely labeled row for the final fit.
X_labeled = X.loc[labeled_mask]
y_labeled = y.loc[labeled_mask].astype(int)
X_full_train = pd.concat([X_labeled, X_pseudo])
y_full_train = pd.concat([y_labeled, y_pseudo])

c:\Users\regan\AppData\Local\Programs\Python\Python310\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 2219, number of negative: 21387
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019739 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 38088
[LightGBM] [Info] Number of data points in the train set: 23606, number of used features: 165
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.094002 -> initscore=-2.265727
[LightGBM] [Info] Start training from score -2.265727
Training until validation scores don't improve for 50 rounds
[50]	train's auc: 0.999791	val's auc: 0.983382
[100]	train's auc: 0.999948	val's auc: 0.990225
[150]	train's auc: 0.999978	val's auc: 0.993693
[200]	train's auc: 0.999997	val's auc: 0.995026
[250]	train's auc: 1	val's auc: 0.995596
[300]	train's auc: 1	val's auc: 0.995768
[350]	train's auc: 1	val's auc: 0.996018
[400]	train's auc: 1	val's auc: 0.996306
[450]	train's auc: 1	val's auc: 0.996411
[500]	train's auc: 1	val's auc: 0.996627
[550]	t

c:\Users\regan\AppData\Local\Programs\Python\Python310\lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 4729, number of negative: 93207
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.190340 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 38857
[LightGBM] [Info] Number of data points in the train set: 97936, number of used features: 166
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.048287 -> initscore=-2.981109
[LightGBM] [Info] Start training from score -2.981109
Training until validation scores don't improve for 50 rounds
[50]	train_augmented's auc: 0.999515	val's auc: 0.970143
Early stopping, best iteration is:
[8]	train_augmented's auc: 0.996801	val's auc: 0.980619

---> Temporal Validation ROC-AUC: 0.98062


In [25]:
# Train the final model on every genuine label plus high-confidence pseudo-labels.
model.fit(X_full_train, y_full_train)
test_preds = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'index': test_df['index'],
    'target': test_preds
})
submission.to_csv('submission_pseudolabelled.csv', index=False)
print(f'Saved submission_pseudolabelled.csv with {len(submission)} predictions.')

[LightGBM] [Info] Number of positive: 6154, number of negative: 99411
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.169782 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 39004
[LightGBM] [Info] Number of data points in the train set: 105565, number of used features: 166
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.058296 -> initscore=-2.782160
[LightGBM] [Info] Start training from score -2.782160
Saved submission_pseudolabelled.csv with 15329 predictions.
